In [1]:
import pandas as pd
from urllib.request import urlopen

df = pd.read_csv('data/raw/qcew-2023-2025q4.csv')

In [2]:
# OPTIONAL: Load data from API if available

'''
base = 'https://publish.data.ca.gov/api/3/action/datastore_search?resource_id=119eef38-3b59-499f-8f7c-9bea4768469d'

try:
    # Load rows from the API in chunks of 1000 until all rows are loaded
    with urlopen(f'{base}&limit=1') as response:
        total = json.load(response)['result']['total']

    records = []
    while len(records) < total:
        remaining = total - len(records)
        with urlopen(f'{base}&limit={remaining}&offset={len(records)}') as response:
            records += json.load(response)['result']['records']

    df = pd.DataFrame(records)
    print('Successfully loaded data from API')

    df = df.rename(columns={
        'unsafe_1st_month_emp': '1st Month Emp',
        'unsafe_2nd_month_emp': '2nd Month Emp',
        'unsafe_3rd_month_emp': '3rd Month Emp',
        'total_wages__all_workers_': 'Total Wages (All Workers)'
    })
    
except Exception as e:
    print(f'Error loading data from API: {e}')
    print ('Falling back to local CSV file...')
    df = pd.read_csv('data/raw/qcew-2023-2025q4.csv')
''';


In [3]:
# Confirm all rows were loaded
total = len(df)
print(f'{len(df):}/{total:} rows loaded')

869884/869884 rows loaded


In [4]:
# Various checks on the data

summary = pd.DataFrame({'dtype': df.dtypes, 'nulls': df.isnull().sum()})
print(summary)
print(f'\nDuplicate rows: {df.duplicated().sum()}')
print()
print(df['Ownership'].value_counts())
print()
print(f'{df["Industry Name"].nunique()} industries')
print()
print(df['Industry Name'].value_counts())

                              dtype  nulls
Area Type                    object      0
Area Name                    object      0
Year                          int64      0
Time Period                  object      0
Ownership                    object      0
NAICS Level                   int64      0
NAICS Code                   object      0
Industry Name                object      0
Establishments                int64      0
Average Monthly Employment    int64      0
1st Month Emp                 int64      0
2nd Month Emp                 int64      0
3rd Month Emp                 int64      0
Total Wages (All Workers)   float64      0
Average Weekly Wages          int64      0

Duplicate rows: 0

Ownership
Private               619790
Local Government      140849
Federal Government     66582
State Government       41718
Total Covered            900
Total Government          30
Total U.I. Covered        15
Name: count, dtype: int64

1507 industries

Industry Name
Public Administration

In [5]:
print(df['NAICS Level'].dtype)
print(df['NAICS Level'].value_counts())

int64
NAICS Level
6    297940
5    257052
4    165529
3     84514
2     54994
1      5325
0      4530
Name: count, dtype: int64


In [6]:
# Limit data to private, annual data

df = df[df['Ownership'] == 'Private']
df = df[df['Time Period'] == 'Annual']
print(f'\n{len(df):}/{total:} rows remaining after filtering for private, annual data')

# drop unnecessary columns
df = df.drop(columns=['1st Month Emp', '2nd Month Emp', '3rd Month Emp'])


122410/869884 rows remaining after filtering for private, annual data


In [11]:
# Split data by NAICS level

df_4 = df[df['NAICS Level'] == 4]
print(f'{len(df_4)} level 4 industries')
df_6 = df[df['NAICS Level'] == 6]
print(f'{len(df_6)} level 6 industries')

# Output to CSV files
df_4.to_csv('data/clean/level_4.csv', index=False)
df_6.to_csv('data/clean/level_6.csv', index=False)

23908 level 4 industries
45415 level 6 industries
